# WC_BADGE_PRODUCT_D ETL - ODI to Databricks Migration

**Original Package:** WC_BADGE_PRODUCT_D Load

**Source Schema:** `workspace.PRXBI_TS`

**Target Schema:** `workspace.PRXBI_DW`

---

## Step 1: Define Widgets/Parameters



In [0]:
%sql
-- Create widgets for parameters
CREATE WIDGET TEXT ETL_JOB_TYPE DEFAULT 'EOD';
CREATE WIDGET TEXT DATASOURCE_NUM_ID DEFAULT '380';
CREATE WIDGET TEXT ETL_PROC_WID DEFAULT '1';

---

## Step 2: Get ETL Parameters

In [0]:
%sql
-- Get last extract time
CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS
SELECT etl_last_extract_time 
FROM workspace.PRXBI_DW.wc_etl_parameters 
WHERE ETL_JOB_TYPE = '${ETL_JOB_TYPE}';

In [0]:
%sql
-- Get current extract time
CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time AS
SELECT etl_current_extract_time 
FROM workspace.PRXBI_DW.wc_etl_parameters 
WHERE ETL_JOB_TYPE = '${ETL_JOB_TYPE}';

In [0]:
%sql
-- Get ROW_WID for ETL parameters
CREATE OR REPLACE TEMPORARY VIEW v_etl_row_wid AS
SELECT ROW_WID 
FROM workspace.PRXBI_DW.wc_etl_parameters 
WHERE ETL_JOB_TYPE = '${ETL_JOB_TYPE}';

In [0]:
%sql
-- Display ETL parameters
SELECT 
    'Last Extract Time' AS parameter,
    etl_last_extract_time AS value
FROM v_etl_last_extract_time
UNION ALL
SELECT 
    'Current Extract Time' AS parameter,
    etl_current_extract_time AS value
FROM v_etl_current_extract_time
UNION ALL
SELECT 
    'ETL ROW_WID' AS parameter,
    CAST(ROW_WID AS STRING) AS value
FROM v_etl_row_wid;

parameter,value
Last Extract Time,2026-01-07T00:00:00.000Z
Current Extract Time,2026-01-07T00:00:00.000Z
ETL ROW_WID,+21937-01-01T00:00:00.000Z


---

## Step 3: Create Staging Table (C$_0FILTER)

In [0]:
%sql
-- Drop staging table if exists
DROP TABLE IF EXISTS workspace.PRXBI_DW.c_0filter_stg;

In [0]:
%sql
-- Create staging table
CREATE TABLE workspace.PRXBI_DW.c_0filter_stg (
    NAME STRING,
    PRICE DECIMAL(25,5),
    STATUS INT,
    VISIBILITY INT,
    WEIGHT DECIMAL(38,0),
    ID STRING,
    PARENT_SKU STRING,
    PARENT_RX_EBS_PRODUCT_CODE STRING,
    CREATED_AT TIMESTAMP,
    UPDATED_AT TIMESTAMP,
    STOCK_QTY DECIMAL(38,0),
    RX_LINK_TYPE STRING
)
USING DELTA;


Executing subquery: -- Drop integration/flow table
DROP TABLE IF EXISTS workspace.PRXBI_DW.i_wc_badge_product_d_flow.
Executing subquery: -- Drop staging table
DROP TABLE IF EXISTS workspace.PRXBI_DW.c_0filter_stg.
Executing subquery: -- Final validation - show summary statistics
SELECT 
    'WC_BADGE_PRODUCT_D' AS table_name,
    COUNT(*) AS total_records,
    COUNT(DISTINCT INTEGRATION_ID) AS unique_integration_ids,
    MAX(W_UPDATE_DT) AS last_update_time,
    'ETL Completed Successfully' AS status
FROM workspace.PRXBI_DW.wc_badge_product_d
WHERE DATASOURCE_NUM_ID = '380'.
Executing subquery: -- Show sample of recently updated records
SELECT 
    ID,
    SKU,
    NAME,
    PRICE,
    STATUS,
    W_UPDATE_DT,
    ETL_PROC_WID
FROM workspace.PRXBI_DW.wc_badge_product_d
WHERE DATASOURCE_NUM_ID = '380'
ORDER BY W_UPDATE_DT DESC
LIMIT 10.
Executing subquery: -- ROW_WID BIGINT GENERATED ALWAYS AS IDENTITY


drop TABLE workspace.PRXBI_DW.WC_BADGE_DETAILS_D 
-- (
--   ROW_WID BIGINT GENERA

---

## Step 4: Extract and Load Data into Staging (with ROW_NUMBER deduplication)

In [0]:
%sql
-- Extract distinct products with row number ranking
INSERT INTO workspace.PRXBI_DW.c_0filter_stg
SELECT 
    DISTINCT_PRODUCT_D.NAME,
    DISTINCT_PRODUCT_D.PRICE,
    DISTINCT_PRODUCT_D.STATUS,
    DISTINCT_PRODUCT_D.VISIBILITY,
    DISTINCT_PRODUCT_D.WEIGHT,
    DISTINCT_PRODUCT_D.ID,
    DISTINCT_PRODUCT_D.PARENT_SKU,
    DISTINCT_PRODUCT_D.PARENT_RX_EBS_PRODUCT_CODE,
    DISTINCT_PRODUCT_D.CREATED_AT,
    DISTINCT_PRODUCT_D.UPDATED_AT,
    DISTINCT_PRODUCT_D.STOCK_QTY,
    DISTINCT_PRODUCT_D.RX_LINK_TYPE
FROM (
    SELECT 
        WC_MERCURY_PRODUCT_TS.NAME,
        WC_MERCURY_PRODUCT_TS.PARENT_PRICE AS PRICE,
        WC_MERCURY_PRODUCT_TS.STATUS,
        WC_MERCURY_PRODUCT_TS.VISIBILITY,
        WC_MERCURY_PRODUCT_TS.WEIGHT,
        WC_MERCURY_PRODUCT_TS.INT_INSERT_DATE,
        WC_MERCURY_PRODUCT_TS.INT_UPDATE_DATE,
        ROW_NUMBER() OVER(
            PARTITION BY WC_MERCURY_PRODUCT_TS.ID 
            ORDER BY WC_MERCURY_PRODUCT_TS.INT_INSERT_DATE DESC
        ) AS RNK,
        WC_MERCURY_PRODUCT_TS.ID,
        WC_MERCURY_PRODUCT_TS.SKU,
        WC_MERCURY_PRODUCT_TS.PARENT_SKU,
        WC_MERCURY_PRODUCT_TS.PARENT_PRICE,
        WC_MERCURY_PRODUCT_TS.PARENT_RX_EBS_PRODUCT_CODE,
        WC_MERCURY_PRODUCT_TS.CREATED_AT,
        WC_MERCURY_PRODUCT_TS.UPDATED_AT,
        WC_MERCURY_PRODUCT_TS.STOCK_QTY,
        WC_MERCURY_PRODUCT_TS.RX_LINK_TYPE
    FROM workspace.PRXBI_TS.wc_mercury_product_ts WC_MERCURY_PRODUCT_TS
) DISTINCT_PRODUCT_D
WHERE DISTINCT_PRODUCT_D.RNK = 1;

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
-- Show staging record count
SELECT COUNT(*) AS staging_record_count 
FROM workspace.PRXBI_DW.c_0filter_stg;

staging_record_count
2


---

## Step 5: Create Integration/Flow Table (I$_WC_BADGE_PRODUCT_D)

In [0]:
%sql
-- Drop integration table if exists
DROP TABLE IF EXISTS workspace.PRXBI_DW.i_wc_badge_product_d_flow;

In [0]:
%sql
-- Create integration/flow table
CREATE TABLE workspace.PRXBI_DW.i_wc_badge_product_d_flow (
    ROW_WID DECIMAL(10,0),
    ID STRING,
    SKU STRING,
    NAME STRING,
    PRICE DECIMAL(25,5),
    STATUS INT,
    VISIBILITY INT,
    WEIGHT DECIMAL(38,0),
    STOCK_QTY DECIMAL(38,0),
    INTEGRATION_ID STRING,
    DATASOURCE_NUM_ID STRING,
    W_INSERT_DT TIMESTAMP,
    W_UPDATE_DT TIMESTAMP,
    ETL_PROC_WID DECIMAL(10,0),
    PARENT_RX_EBS_PRODUCT_CODE STRING,
    CHANGED_ON_DT TIMESTAMP,
    CREATED_ON_DT TIMESTAMP,
    RX_LINK_TYPE STRING,
    IND_UPDATE STRING
)
USING DELTA;

---

## Step 6: Detect New Records (Insert Detection - NOT EXISTS Strategy)

In [0]:
%sql
-- Insert new records into flow table with change detection
-- Detection Strategy: NOT EXISTS - inserts only records that don't exist or have changed
INSERT INTO workspace.PRXBI_DW.i_wc_badge_product_d_flow (
    ID,
    SKU,
    NAME,
    PRICE,
    STATUS,
    VISIBILITY,
    WEIGHT,
    STOCK_QTY,
    INTEGRATION_ID,
    DATASOURCE_NUM_ID,
    W_INSERT_DT,
    W_UPDATE_DT,
    PARENT_RX_EBS_PRODUCT_CODE,
    CHANGED_ON_DT,
    CREATED_ON_DT,
    RX_LINK_TYPE,
    IND_UPDATE
)
SELECT 
    S.ID,
    S.SKU,
    S.NAME,
    S.PRICE,
    S.STATUS,
    S.VISIBILITY,
    S.WEIGHT,
    S.STOCK_QTY,
    S.INTEGRATION_ID,
    S.DATASOURCE_NUM_ID,
    S.W_INSERT_DT,
    S.W_UPDATE_DT,
    S.PARENT_RX_EBS_PRODUCT_CODE,
    S.CHANGED_ON_DT,
    S.CREATED_ON_DT,
    S.RX_LINK_TYPE,
    S.IND_UPDATE
FROM (
    SELECT 
        FILTER_A.ID AS ID,
        FILTER_A.PARENT_SKU AS SKU,
        FILTER_A.NAME AS NAME,
        FILTER_A.PRICE AS PRICE,
        FILTER_A.STATUS AS STATUS,
        FILTER_A.VISIBILITY AS VISIBILITY,
        FILTER_A.WEIGHT AS WEIGHT,
        FILTER_A.STOCK_QTY AS STOCK_QTY,
        FILTER_A.ID AS INTEGRATION_ID,
        '${DATASOURCE_NUM_ID}' AS DATASOURCE_NUM_ID,
        CURRENT_TIMESTAMP() AS W_INSERT_DT,
        CURRENT_TIMESTAMP() AS W_UPDATE_DT,
        FILTER_A.PARENT_RX_EBS_PRODUCT_CODE AS PARENT_RX_EBS_PRODUCT_CODE,
        FILTER_A.UPDATED_AT AS CHANGED_ON_DT,
        FILTER_A.CREATED_AT AS CREATED_ON_DT,
        FILTER_A.RX_LINK_TYPE AS RX_LINK_TYPE,
        'I' AS IND_UPDATE
    FROM workspace.PRXBI_DW.c_0filter_stg FILTER_A
    WHERE (1=1)
) S
WHERE NOT EXISTS (
    SELECT 1 
    FROM workspace.PRXBI_DW.wc_badge_product_d T
    WHERE T.INTEGRATION_ID = S.INTEGRATION_ID
        AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID
        AND (
            (T.ID = S.ID OR (T.ID IS NULL AND S.ID IS NULL))
            AND (T.SKU = S.SKU OR (T.SKU IS NULL AND S.SKU IS NULL))
            AND (T.NAME = S.NAME OR (T.NAME IS NULL AND S.NAME IS NULL))
            AND (T.PRICE = S.PRICE OR (T.PRICE IS NULL AND S.PRICE IS NULL))
            AND (T.STATUS = S.STATUS OR (T.STATUS IS NULL AND S.STATUS IS NULL))
            AND (T.VISIBILITY = S.VISIBILITY OR (T.VISIBILITY IS NULL AND S.VISIBILITY IS NULL))
            AND (T.WEIGHT = S.WEIGHT OR (T.WEIGHT IS NULL AND S.WEIGHT IS NULL))
            AND (T.STOCK_QTY = S.STOCK_QTY OR (T.STOCK_QTY IS NULL AND S.STOCK_QTY IS NULL))
            AND (T.W_UPDATE_DT = S.W_UPDATE_DT OR (T.W_UPDATE_DT IS NULL AND S.W_UPDATE_DT IS NULL))
            AND (T.PARENT_RX_EBS_PRODUCT_CODE = S.PARENT_RX_EBS_PRODUCT_CODE 
                OR (T.PARENT_RX_EBS_PRODUCT_CODE IS NULL AND S.PARENT_RX_EBS_PRODUCT_CODE IS NULL))
            AND (T.CHANGED_ON_DT = S.CHANGED_ON_DT OR (T.CHANGED_ON_DT IS NULL AND S.CHANGED_ON_DT IS NULL))
            AND (T.CREATED_ON_DT = S.CREATED_ON_DT OR (T.CREATED_ON_DT IS NULL AND S.CREATED_ON_DT IS NULL))
            AND (T.RX_LINK_TYPE = S.RX_LINK_TYPE OR (T.RX_LINK_TYPE IS NULL AND S.RX_LINK_TYPE IS NULL))
        )
);

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
-- Show new records detected
SELECT COUNT(*) AS new_records_detected 
FROM workspace.PRXBI_DW.i_wc_badge_product_d_flow 
WHERE IND_UPDATE = 'I';

new_records_detected
2


---

## Step 7: Mark Records for Update

In [0]:
%sql
-- Mark records as UPDATE when they already exist in target
UPDATE workspace.PRXBI_DW.i_wc_badge_product_d_flow F
SET IND_UPDATE = 'U'
WHERE EXISTS (
    SELECT 1
    FROM workspace.PRXBI_DW.wc_badge_product_d T
    WHERE T.INTEGRATION_ID   = F.INTEGRATION_ID
      AND T.DATASOURCE_NUM_ID = F.DATASOURCE_NUM_ID
);


num_affected_rows
0


In [0]:
%sql
-- Show update vs insert breakdown
SELECT 
    IND_UPDATE,
    COUNT(*) AS record_count
FROM workspace.PRXBI_DW.i_wc_badge_product_d_flow
GROUP BY IND_UPDATE;

IND_UPDATE,record_count
I,2


---

## Step 8: Perform MERGE Operation (Update + Insert)

In [0]:
%sql
-- MERGE into target table using Databricks MERGE syntax
MERGE INTO workspace.PRXBI_DW.wc_badge_product_d AS T
USING workspace.PRXBI_DW.i_wc_badge_product_d_flow AS S
ON T.INTEGRATION_ID = S.INTEGRATION_ID 
   AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID
WHEN MATCHED AND S.IND_UPDATE = 'U' THEN UPDATE SET
    T.ID = S.ID,
    T.SKU = S.SKU,
    T.NAME = S.NAME,
    T.PRICE = S.PRICE,
    T.STATUS = S.STATUS,
    T.VISIBILITY = S.VISIBILITY,
    T.WEIGHT = S.WEIGHT,
    T.STOCK_QTY = S.STOCK_QTY,
    T.W_UPDATE_DT = S.W_UPDATE_DT,
    T.PARENT_RX_EBS_PRODUCT_CODE = S.PARENT_RX_EBS_PRODUCT_CODE,
    T.CHANGED_ON_DT = S.CHANGED_ON_DT,
    T.CREATED_ON_DT = S.CREATED_ON_DT,
    T.RX_LINK_TYPE = S.RX_LINK_TYPE,
    T.ETL_PROC_WID = ${ETL_PROC_WID}
WHEN NOT MATCHED AND S.IND_UPDATE = 'I' THEN INSERT (
    ID,
    SKU,
    NAME,
    PRICE,
    STATUS,
    VISIBILITY,
    WEIGHT,
    STOCK_QTY,
    INTEGRATION_ID,
    DATASOURCE_NUM_ID,
    W_INSERT_DT,
    W_UPDATE_DT,
    PARENT_RX_EBS_PRODUCT_CODE,
    CHANGED_ON_DT,
    CREATED_ON_DT,
    RX_LINK_TYPE,
    ETL_PROC_WID
) VALUES (
    S.ID,
    S.SKU,
    S.NAME,
    S.PRICE,
    S.STATUS,
    S.VISIBILITY,
    S.WEIGHT,
    S.STOCK_QTY,
    S.INTEGRATION_ID,
    S.DATASOURCE_NUM_ID,
    S.W_INSERT_DT,
    S.W_UPDATE_DT,
    S.PARENT_RX_EBS_PRODUCT_CODE,
    S.CHANGED_ON_DT,
    S.CREATED_ON_DT,
    S.RX_LINK_TYPE,
    ${ETL_PROC_WID}
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,0,0,2


---

## Step 9: Optimize Target Table (Optional - replaces Oracle stats gathering)

In [0]:
%sql
-- Optimize Delta table for better query performance
-- This is Databricks equivalent of Oracle's dbms_stats.gather_table_stats
OPTIMIZE workspace.PRXBI_DW.wc_badge_product_d;

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1767870923220, 1767870924692, 8, 0, null, List(0, 0), null, 22, 22, 0, 0, null)"


In [0]:
%sql
-- Optional: Z-order by frequently filtered columns
OPTIMIZE workspace.PRXBI_DW.wc_badge_product_d
ZORDER BY (INTEGRATION_ID, DATASOURCE_NUM_ID);

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 5733), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1767870926563, 1767870927998, 8, 0, null, List(0, 0), null, 22, 22, 0, 0, null)"


---

## Step 10: Cleanup - Drop Temporary Tables

In [0]:
%sql
-- Drop integration/flow table
DROP TABLE IF EXISTS workspace.PRXBI_DW.i_wc_badge_product_d_flow;

In [0]:
%sql
-- Drop staging table
DROP TABLE IF EXISTS workspace.PRXBI_DW.c_0filter_stg;

---

## Step 11: Validation & Summary

In [0]:
%sql
-- Final validation - show summary statistics
SELECT 
    'WC_BADGE_PRODUCT_D' AS table_name,
    COUNT(*) AS total_records,
    COUNT(DISTINCT INTEGRATION_ID) AS unique_integration_ids,
    MAX(W_UPDATE_DT) AS last_update_time,
    'ETL Completed Successfully' AS status
FROM workspace.PRXBI_DW.wc_badge_product_d
WHERE DATASOURCE_NUM_ID = '${DATASOURCE_NUM_ID}';

table_name,total_records,unique_integration_ids,last_update_time,status
WC_BADGE_PRODUCT_D,2,2,2026-01-08,ETL Completed Successfully


In [0]:
%sql
-- Show sample of recently updated records
SELECT 
    ID,
    SKU,
    NAME,
    PRICE,
    STATUS,
    W_UPDATE_DT,
    ETL_PROC_WID
FROM workspace.PRXBI_DW.wc_badge_product_d
WHERE DATASOURCE_NUM_ID = '${DATASOURCE_NUM_ID}'
ORDER BY W_UPDATE_DT DESC
LIMIT 10;

ID,SKU,NAME,PRICE,STATUS,W_UPDATE_DT,ETL_PROC_WID
80332,PACKAGE-MIPCM-3500411,Visitor early bird rate,0,1,2026-01-08,1
84076,PACKAGE-EDIX-38100022,Visitor - Government,0,1,2026-01-08,1


---

## Notes on Conversion

### Key Changes from ODI to Databricks:

1. **Sequences Removed**: `WC_BADGE_PRODUCT_D_SEQ.NEXTVAL` removed. If ROW_WID is needed, use `ROW_NUMBER()` or auto-increment identity columns in table definition.

2. **Oracle Hints Removed**: `/*+ append */` hints are not needed in Spark SQL.

3. **SYSTIMESTAMP → CURRENT_TIMESTAMP()**: Standard Spark SQL function.

4. **NOLOGGING Removed**: Not applicable in Delta Lake.

5. **Index Creation Skipped**: Delta Lake handles indexing automatically. OPTIMIZE and Z-ORDER commands replace traditional indexing.

6. **dbms_stats Replaced**: `OPTIMIZE` command handles statistics and compaction.

7. **Separate UPDATE + INSERT → MERGE**: Databricks MERGE combines both operations efficiently.

8. **Schema References**: 
   - `PRXBI_DW_SEP` → `workspace.PRXBI_DW`
   - `PRXBI_TS_SEP` → `workspace.PRXBI_TS`

### Manual Actions Required:

1. **ROW_WID Column**: If ROW_WID needs to be sequential and persisted, add identity column to target table definition:

In [0]:
%sql
-- ROW_WID BIGINT GENERATED ALWAYS AS IDENTITY



2. **ETL Parameters Table**: Ensure `wc_etl_parameters` exists in target schema with proper structure.

3. **Target Table Creation**: If `wc_badge_product_d` doesn't exist, create it with proper schema before running this notebook.

---

## Remove Widgets (Optional - run at end)

In [0]:
%sql
-- Remove widgets at the end of the job
-- REMOVE WIDGET ETL_JOB_TYPE;
-- REMOVE WIDGET DATASOURCE_NUM_ID;
-- REMOVE WIDGET ETL_PROC_WID;